# 02 - Client data integration

This notebook prepares and validates the optional integration of `clientes.csv` with the transaction data. It does **not** yet split the data, preprocess variables or train a model. Those decisions will be agreed by the group before implementation.

In [1]:
from pathlib import Path

import pandas as pd

TARGET = 'repeat_purchase_90d'

# Works whether Jupyter starts in the repository root or in notebooks/.
PROJECT_DIR = Path.cwd().resolve()
if not (PROJECT_DIR / 'data').exists():
    PROJECT_DIR = PROJECT_DIR.parent
DATA_DIR = PROJECT_DIR / 'data'

## 1. Load the official files

`train.csv` contains the labelled purchase occasions. `test.csv` contains later unlabelled purchase occasions. `clientes.csv` contains one optional row of customer-level information per `customer_id`.

In [2]:
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
clientes = pd.read_csv(DATA_DIR / 'clientes.csv')

print(f'Train: {train.shape}')
print(f'Test: {test.shape}')
print(f'Clientes: {clientes.shape}')

Train: (6534, 39)
Test: (986, 38)
Clientes: (1143, 13)


## 2. Variables selected for the first client-data experiment

The following variables are candidate predictors because they are known at the time of purchase, are not direct customer identifiers and may capture geographic or customer-cohort patterns. `customer_id` is retained only as the key used for the merge.

For now we exclude `company_name`, `address` and `street` because they are almost unique per customer; `customer_type` because it is entirely missing; and `city` and `postal_code` because their high cardinality should be tested separately later.

In [3]:
CLIENT_FEATURES_CORE = [
    'country_or_region',
    'commercial_region',
    'island',
    'municipality',
    'postal_area',
    'customer_since_year',
]

CLIENT_JOIN_COLUMNS = ['customer_id'] + CLIENT_FEATURES_CORE
display(clientes[CLIENT_JOIN_COLUMNS].head())

,customer_id,country_or_region,commercial_region,island,municipality,postal_area,customer_since_year
0,842570,Country CT11: Land of The Final Version,Region RG37: We Will See Later,NaN,NaN,NaN,2018
1,770941,Country CT9: Land of The Final Version,NaN,NaN,NaN,NaN,2021
2,827053,Country CT10: Land of The Final Version,Region RG17: We Will See Later,Island IS2: The Pivot Archipelago,Municipality MU14: Council of Last Minute Edits,Postal area PA271: Hidden Sheet Sector,2016
3,494648,Country CT10: Land of The Final Version,Region RG17: We Will See Later,Island IS2: The Pivot Archipelago,Municipality MU14: Council of Last Minute Edits,Postal area PA271: Hidden Sheet Sector,2015
4,429291,Country CT10: Land of The Final Version,Region RG17: We Will See Later,Island IS2: The Pivot Archipelago,Municipality MU14: Council of Last Minute Edits,Postal area PA271: Hidden Sheet Sector,2015


## 3. Safe merge

The checks below confirm that `clientes.csv` has one row per customer, every customer in `train.csv` and `test.csv` has a match, and the merge does not add or remove purchase rows. The original CSV files are never changed; `train_with_clients` and `test_with_clients` exist only in memory while the notebook runs.

In [4]:
assert clientes['customer_id'].is_unique, 'clientes.csv must have one row per customer_id.'
assert train['customer_id'].isin(clientes['customer_id']).all(), 'Some train customers do not match clientes.csv.'
assert test['customer_id'].isin(clientes['customer_id']).all(), 'Some test customers do not match clientes.csv.'

train_with_clients = train.merge(
    clientes[CLIENT_JOIN_COLUMNS],
    on='customer_id',
    how='left',
    validate='many_to_one',
)
test_with_clients = test.merge(
    clientes[CLIENT_JOIN_COLUMNS],
    on='customer_id',
    how='left',
    validate='many_to_one',
)

assert len(train_with_clients) == len(train), 'The train merge changed the number of rows.'
assert len(test_with_clients) == len(test), 'The test merge changed the number of rows.'

print('Safe merge confirmed.')
print(f'Train rows before/after merge: {len(train):,} / {len(train_with_clients):,}')
print(f'Test rows before/after merge: {len(test):,} / {len(test_with_clients):,}')
display(train_with_clients[CLIENT_JOIN_COLUMNS + [TARGET]].head())

Safe merge confirmed.
Train rows before/after merge: 6,534 / 6,534
Test rows before/after merge: 986 / 986


,customer_id,country_or_region,commercial_region,island,municipality,postal_area,customer_since_year,repeat_purchase_90d
0,660784,Country CT10: Land of The Final Version,Region RG25: We Will See Later,NaN,Municipality MU89: Council of Last Minute Edits,Postal area PA228: Hidden Sheet Sector,2024,1
1,189146,Country CT10: Land of The Final Version,Region RG17: We Will See Later,Island IS8: The Pivot Archipelago,Municipality MU11: Council of Last Minute Edits,Postal area PA260: Hidden Sheet Sector,2015,1
2,103940,Country CT10: Land of The Final Version,Region RG23: We Will See Later,NaN,Municipality MU70: Council of Last Minute Edits,Postal area PA84: Hidden Sheet Sector,2017,1
3,512927,Country CT10: Land of The Final Version,Region RG17: We Will See Later,Island IS7: The Pivot Archipelago,Municipality MU12: Council of Last Minute Edits,Postal area PA253: Hidden Sheet Sector,2018,0
4,122356,Country CT10: Land of The Final Version,Region RG31: We Will See Later,NaN,Municipality MU142: Council of Last Minute Edits,Postal area PA136: Hidden Sheet Sector,2017,1


## Next group decisions

The group should next agree on the validation strategy, missing-value treatment, feature engineering and the single classification model required for the intermediate delivery. Once those choices are made, a later notebook can compare transaction-only data with this merged version under the same evaluation procedure.